# Práctica Final de Sección
## Extracción de Características

1. Escalar el código proporcionado de obtención de esquinas (Leer mucho en una sola corrida y obtener todas las métricas)
* Obtener la densidad absoluta de cada tela dentro de la carpeta img/telas.
* Hacer una tabla de tela por densidad
* Seleccionar 4 telas y hacer la comparativa:
* Imagen original vs Imagen Con sus Esquinas

### Reporte
1. Anexar capturas de todas las imágenes procesadas.
2. Actualizar su repositorio Github con el código empleado para la práctica. Anexar el URL en el reporte.


In [1]:
import cv2
import os
def encontrar_esquinas(img):
    detector = cv2.FastFeatureDetector_create(
        threshold = 60,
        nonmaxSuppression = True,
        type = cv2.FastFeatureDetector_TYPE_7_12
    )
    return detector.detect(img, mask=None)

def dibujar_puntos(img, pts):
    return cv2.drawKeypoints(img, pts, None, color=(0,0,255))

In [3]:
import os
import cv2

ruta = "img/telas/"
imagenes = os.listdir(ruta)

resultados = []

for archivo in imagenes:
    if archivo.endswith(".jpg") or archivo.endswith(".png") or archivo.endswith(".jpeg"):
        
        img = cv2.imread(ruta + archivo)
        
        if img is None:
            continue
        
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Esquinas
        pts = encontrar_esquinas(img_gray)
        img_pts = dibujar_puntos(img.copy(), pts)
        
        # Densidad
        densidad = (img_gray > 0).sum() / img_gray.size * 100
        
        # Guardar datos
        resultados.append([archivo, densidad, len(pts)])
        
        print("Procesando:", archivo)
        print("Densidad:", densidad)
        print("Esquinas:", len(pts))
        
        # Comparación en una sola ventana
        comparacion = cv2.hconcat([img, img_pts])
        cv2.imshow("Comparacion", comparacion)
        cv2.waitKey(1000)

cv2.destroyAllWindows()
print("\nTabla de resultados:")
for r in resultados:
    print(r)

Procesando: Corduroy.jpg
Densidad: 100.0
Esquinas: 3
Procesando: Cotton-Flannel.jpg
Densidad: 100.0
Esquinas: 1
Procesando: Cotton-Jersey.jpg
Densidad: 100.0
Esquinas: 1
Procesando: Faux-Fur.jpg
Densidad: 100.0
Esquinas: 24
Procesando: Felt-Fabric.jpg
Densidad: 100.0
Esquinas: 0
Procesando: Linen.jpg
Densidad: 100.0
Esquinas: 8
Procesando: Oxford.jpg
Densidad: 100.0
Esquinas: 11650
Procesando: Polyester.jpg
Densidad: 100.0
Esquinas: 2378
Procesando: Satin.jpg
Densidad: 100.0
Esquinas: 0
Procesando: Sherpa.jpg
Densidad: 100.0
Esquinas: 0
Procesando: Silk.jpg
Densidad: 100.0
Esquinas: 215
Procesando: Terry.jpg
Densidad: 100.0
Esquinas: 3369
Procesando: Tweed.jpg
Densidad: 100.0
Esquinas: 281
Procesando: Velvet.jpg
Densidad: 100.0
Esquinas: 0

Tabla de resultados:
['Corduroy.jpg', np.float64(100.0), 3]
['Cotton-Flannel.jpg', np.float64(100.0), 1]
['Cotton-Jersey.jpg', np.float64(100.0), 1]
['Faux-Fur.jpg', np.float64(100.0), 24]
['Felt-Fabric.jpg', np.float64(100.0), 0]
['Linen.jpg', np.f

2. Obtener la transformación afín usando los tres mejores matches de esquinas con lena (img/lena_rotada.jpeg).
* Usar getAffineTransform y warpAffine con los tres mejores matches

In [6]:
import cv2
import numpy as np

# Leer imágenes
img1 = cv2.imread("img/lena.jpeg")  # tu imagen
img2 = cv2.imread("img/lena_rotada.jpeg")

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# Detector ORB
orb = cv2.ORB_create()

kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

# Matching
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

# Ordenar mejores matches
matches = sorted(matches, key=lambda x: x.distance)

# Tomar los 3 mejores
best_matches = matches[:3]

# Obtener puntos
pts1 = []
pts2 = []

for m in best_matches:
    pts1.append(kp1[m.queryIdx].pt)
    pts2.append(kp2[m.trainIdx].pt)

pts1 = np.float32(pts1)
pts2 = np.float32(pts2)

# Transformación afín
M = cv2.getAffineTransform(pts1, pts2)

# =========================================
# Aplicar la matriz a puntos
# =========================================
pts1_hom = np.hstack([pts1, np.ones((3,1))])  # coordenadas homogéneas
pts_transformados = np.dot(M, pts1_hom.T).T

print("\nPuntos originales:\n", pts1)
print("\nPuntos transformados con M:\n", pts_transformados)
print("\nPuntos destino reales:\n", pts2)

# =========================================
# Aplicar transformación a la imagen
# =========================================
filas, cols = img1.shape[:2]
resultado = cv2.warpAffine(img1, M, (cols, filas))

# =========================================
# Dibujar puntos en imagen
# =========================================
img_puntos = img1.copy()

for p in pts1:
    cv2.circle(img_puntos, tuple(np.int32(p)), 5, (0,255,0), -1)

for p in pts_transformados:
    cv2.circle(img_puntos, tuple(np.int32(p)), 5, (0,0,255), -1)

cv2.imshow("Puntos transformados (verde=original, rojo=transformado)", img_puntos)

# Mostrar resultados
comparacionlena = cv2.hconcat([img1, img2, resultado])
cv2.imshow("Comparacion", comparacionlena)

# Matches visuales
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, best_matches, None)
cv2.imshow("Matches", img_matches)

print("\nMatriz afín:\n", M)

cv2.waitKey(0)
cv2.destroyAllWindows()


Puntos originales:
 [[162.       92.     ]
 [141.      116.     ]
 [ 85.01761  95.38561]]

Puntos transformados con M:
 [[107.          59.        ]
 [124.          86.        ]
 [ 89.16481018 134.78401184]]

Puntos destino reales:
 [[107.       59.     ]
 [124.       86.     ]
 [ 89.16481 134.78401]]

Matriz afín:
 [[  0.2733496    0.94751423 -24.45394452]
 [ -0.97237526   0.27417165 191.3010008 ]]


3. Ejecutar el código de tracking con el video Traffic.jpeg. Ajustar la máscara de detección de esquinas al centro de la avenida para mejorar el tracking sobre los vehículos en movimiento (evitar la detección de esquinas estáticas como señalamientos, edificios, árboles, etc.). Tomar capturas de la ejecución del código y mostrar la máscara utilizada.

In [14]:
import cv2
import numpy as np

# =========================
# Parámetros
# =========================
lk_params = dict(
    winSize=(15, 15),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.01)
)

feature_params = dict(
    maxCorners=200,
    qualityLevel=0.2,
    minDistance=5,
    blockSize=7
)

MAX_ERROR = 20
MAX_DISPLACEMENT = 50
MIN_POINTS_RATIO = 0.2

# =========================
#  NUEVA FUNCIÓN (gamma)
# =========================
def adjust_gamma(image, gamma=1.5):
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255
                      for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(image, table)

# =========================
# Funciones
# =========================
def detect_features(gray, mask=None):
    pts = cv2.goodFeaturesToTrack(gray, mask=mask, **feature_params)
    if pts is not None:
        pts = np.float32(pts)
    return pts

def filter_points(p0, p1, st, err):
    if p1 is None or st is None or err is None:
        return None, None

    st = st.reshape(-1)
    err = err.reshape(-1)

    valid = (st == 1) & (err < MAX_ERROR)

    p0_valid = p0[valid]
    p1_valid = p1[valid]

    displacement = np.linalg.norm(p1_valid - p0_valid, axis=2).reshape(-1)
    drift_mask = displacement < MAX_DISPLACEMENT

    return p0_valid[drift_mask], p1_valid[drift_mask]

# =========================
# Video
# =========================
video = cv2.VideoCapture("img/Traffic.mp4")

ret, old_frame = video.read()
if not ret:
    raise Exception("No se pudo leer el video")

#  PREPROCESAMIENTO INICIAL
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
old_gray = adjust_gamma(old_gray, gamma=1.5)
old_gray = cv2.GaussianBlur(old_gray, (5,5), 0)

# =========================
# Máscara centrada
# =========================
corner_mask = np.zeros_like(old_gray, dtype=np.uint8)
h, w = old_gray.shape

x1 = int(w * 0.4)
x2 = int(w * 0.63)
y1 = int(h * 0.38)
y2 = int(h * 0.63)

corner_mask[y1:y2, x1:x2] = 255

cv2.imshow("Mascara de tracking", corner_mask)

# Detectar puntos iniciales
p0 = detect_features(old_gray, corner_mask)

mask_draw = np.zeros_like(old_frame)
colors = np.random.randint(0, 255, (1000, 3))

print("Presiona 'q' para salir")

while True:
    ret, frame = video.read()

    if not ret:
        print("Reiniciando video...")
        video.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    # =========================
    # PREPROCESAMIENTO (QUITAR SOMBRAS)
    # =========================
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_gray = adjust_gamma(frame_gray, gamma=3)
    frame_gray = cv2.GaussianBlur(frame_gray, (5,5), 0)

    # =========================
    # Optical Flow
    # =========================
    if p0 is not None and len(p0) > 0:
        p1, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray, frame_gray, p0, None, **lk_params
        )

        good_old, good_new = filter_points(p0, p1, st, err)

        if good_new is not None and len(good_new) > 0:

            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()

                mask_draw = cv2.line(mask_draw, (int(c), int(d)),
                                     (int(a), int(b)),
                                     colors[i % len(colors)].tolist(), 2)

                frame = cv2.circle(frame, (int(a), int(b)), 4,
                                   colors[i % len(colors)].tolist(), -1)

            p0 = good_new.reshape(-1, 1, 2)

        else:
            p0 = None

    # =========================
    # Re-detección
    # =========================
    if p0 is None or len(p0) < feature_params['maxCorners'] * MIN_POINTS_RATIO:
        print("Re-detectando puntos...")

        new_pts = detect_features(frame_gray, corner_mask)

        if new_pts is not None:
            if p0 is not None:
                p0 = np.concatenate((p0, new_pts), axis=0)
            else:
                p0 = new_pts

        mask_draw = np.zeros_like(frame)

    # =========================
    # Visualización
    # =========================
    output = cv2.add(frame, mask_draw)

    cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.putText(output, f"Puntos activos: {0 if p0 is None else len(p0)}",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 255, 0), 2)

    cv2.imshow("Optical Flow (Robusto)", output)

    old_gray = frame_gray.copy()

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()


Presiona 'q' para salir
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...


In [29]:
cv2.imshow("IMG", corner_mask)
cv2.waitKey(0)

-1